# PyTorch 自动求导练习

本模块练习计算图、`requires_grad`、`backward()`、梯度累积、`detach()` 和 `no_grad()`。请只修改每题的 TODO 区域。

In [37]:
import torch

torch.manual_seed(42)
print("PyTorch 版本:", torch.__version__)

PyTorch 版本: 2.14.0+cu130


## 练习 1：第一次反向传播 ⭐

创建标量组成的一维 Tensor `x=[1,2,3]`，令它记录梯度。计算 $y=x^2+2x$，将 y 求和得到标量 loss，并反向传播。

In [38]:
# TODO
x = torch.tensor([1,2,3],dtype=torch.float32,requires_grad=True)
y = x**2+2*x
loss = y.sum()
print(x)
print(y)
print(loss)
# 在这里执行反向传播
loss.backward()

assert x.requires_grad
assert loss.ndim == 0
assert torch.allclose(x.grad, torch.tensor([4.0, 6.0, 8.0]))
print("✅ 练习 1 通过")

tensor([1., 2., 3.], requires_grad=True)
tensor([ 3.,  8., 15.], grad_fn=<AddBackward0>)
tensor(26., grad_fn=<SumBackward0>)
✅ 练习 1 通过


## 练习 2：用链式法则检查梯度 ⭐⭐

模型为 `prediction = w * x + b`，损失为预测值与目标值之差的平方。完成前向与反向传播，并观察梯度是否和手算结果一致。

In [39]:
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
x = torch.tensor(3.0)
target = torch.tensor(10.0)

# TODO
prediction = w*x+b
loss = (prediction-target)**2
print(prediction)
print(loss)
# 反向传播
loss.backward()

assert torch.isclose(prediction, torch.tensor(7.0))
assert torch.isclose(loss, torch.tensor(9.0))
assert torch.isclose(w.grad, torch.tensor(-18.0))
assert torch.isclose(b.grad, torch.tensor(-6.0))
print("✅ 练习 2 通过")

tensor(7., grad_fn=<AddBackward0>)
tensor(9., grad_fn=<PowBackward0>)
✅ 练习 2 通过


## 练习 3：非标量输出的反向传播 ⭐⭐⭐

令 `y=x**2`。由于 y 不是标量，请把 `weights` 作为传入梯度，计算加权和对应的梯度。

In [40]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
weights = torch.tensor([1.0, 0.1, 0.01])
y = x ** 2
print(x.grad)
# TODO：对 y 执行带 weights 的反向传播
y.backward(gradient=weights)
print(x.grad)
assert torch.allclose(x.grad, torch.tensor([2.0, 0.4, 0.06]))
print("✅ 练习 3 通过")

None
tensor([2.0000, 0.4000, 0.0600])
✅ 练习 3 通过


## 练习 4：理解梯度累积 ⭐⭐

对同一个叶子 Tensor 连续反向传播两次，记录梯度；然后清零梯度，再反向传播一次。每次都要重新构建计算结果。

In [41]:
w = torch.tensor(2.0, requires_grad=True)

# TODO
# 第一次：loss = w ** 2，反向传播后将梯度的数值复制到 first_grad
loss = w**2
loss.backward()
first_grad = w.grad
# 第二次：再次构建 loss 并反向传播，将累积梯度复制到 accumulated_grad
loss = w**2
loss.backward()
accumulated_grad = w.grad.clone()
# 将 w.grad 原地清零；第三次重新构建 loss 并反向传播
w.grad.zero_()
loss = w**2
loss.backward()
reset_grad = w.grad
print(first_grad)
print(accumulated_grad)
print(reset_grad)
assert torch.isclose(first_grad, torch.tensor(4.0))
assert torch.isclose(accumulated_grad, torch.tensor(8.0))
assert torch.isclose(reset_grad, torch.tensor(4.0))
print("✅ 练习 4 通过")

tensor(4.)
tensor(8.)
tensor(4.)
✅ 练习 4 通过


## 练习 5：停止记录梯度 ⭐⭐

分别使用 `detach()` 和 `torch.no_grad()` 得到不记录梯度的 Tensor。

In [42]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = 3 * x

# TODO
detached_y = y.detach()
with torch.no_grad():
    prediction = 5*x

assert y.requires_grad
assert not detached_y.requires_grad
assert not prediction.requires_grad
assert torch.equal(detached_y, torch.tensor([3.0, 6.0, 9.0]))
assert torch.equal(prediction, torch.tensor([5.0, 10.0, 15.0]))
print("✅ 练习 5 通过")

✅ 练习 5 通过


## 练习 6：手动完成一次梯度下降 ⭐⭐⭐

使用 $y=2x+1$ 的三个样本。计算均方误差、反向传播，然后在 `no_grad` 中用学习率 0.1 更新 w 和 b，最后清空梯度。

In [43]:
x = torch.tensor([1.0, 2.0, 3.0])
target = 2 * x + 1
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
lr = 0.1

# TODO
prediction = w*x+b
loss = torch.mean((prediction-target)**2)
print(loss)
print(w,b)
# 反向传播
loss.backward()
# 在 no_grad 中更新 w 和 b
print(w.grad,b.grad)
with torch.no_grad():
    w -= lr*w.grad
    b -= lr*b.grad
print(w,b)
# 将 w.grad 和 b.grad 设为 None
w.grad = None
b.grad = None
assert torch.isclose(loss, torch.tensor(83 / 3), atol=1e-5)
assert torch.isclose(w, torch.tensor(34 / 15), atol=1e-5)
assert torch.isclose(b, torch.tensor(1.0), atol=1e-5)
assert w.grad is None and b.grad is None
print("✅ 练习 6 通过")

tensor(27.6667, grad_fn=<MeanBackward0>)
tensor(0., requires_grad=True) tensor(0., requires_grad=True)
tensor(-22.6667) tensor(-10.)
tensor(2.2667, requires_grad=True) tensor(1., requires_grad=True)
✅ 练习 6 通过


## 过关标准

你应该能解释：叶子 Tensor、标量 loss、链式法则、梯度为什么会累积，以及训练和推理时为什么需要不同的梯度设置。